# ARC-style builder dry-run

Walks the public `tckdb_client.builders` API through a workflow-shaped
upload — the kind of data an ARC-style pipeline would produce — and
prints what would be sent without performing the live POST unless
`TCKDB_BASE_URL` and `TCKDB_API_KEY` are both set.

**Chemistry:** `CH4 + OH → CH3 + H2O` (H-abstraction). One realistic
conformer per species, one TS, two attached artifacts.

This is **not** an ARC integration. It uses no ARC code; it mocks the
*shape* of an ARC-style submission so the public builder API
(`SourceCalculations`, `upload.summary()`,
`upload.emission_diagnostics()`, `upload.artifact_plan_preview()`) can
be exercised on workflow-shaped data.

### Conformer-boundary policy
This notebook represents **what the workflow stands behind** — one
scientifically meaningful geometry per species — and never the
candidate list the workflow walked past. See
[`docs/conformer_semantic_boundary.md`](../docs/conformer_semantic_boundary.md).

A sibling Python script lives at
[`builder_arc_style_dry_run.py`](builder_arc_style_dry_run.py); the
two stay in lockstep and are covered by matching smoke tests.


## 1 · Imports and shared constants

Mixed Gaussian-opt / ORCA-SP releases mirror realistic ARC-style
pipelines: optimisation + frequency at one LoT, single-point
refinement at a higher LoT.


In [ ]:
from __future__ import annotations

import json
import os
import tempfile
import time
import warnings
from pathlib import Path

from tckdb_client import TCKDBClient
from tckdb_client.builders import (
    Calculation,
    ChemReaction,
    ComputedReactionUpload,
    Geometry,
    Kinetics,
    LevelOfTheory,
    PlannedArtifactUpload,
    SourceCalculations,
    Species,
    SoftwareRelease,
    Statmech,
    Thermo,
    TransitionState,
)

GAUSSIAN = SoftwareRelease(software="Gaussian", version="16", revision="C.01")
ORCA = SoftwareRelease(software="ORCA", version="5.0.4")
LOT_OPT_FREQ = LevelOfTheory(method="wb97xd", basis="def2tzvp")
LOT_SP = LevelOfTheory(method="ccsd(t)-f12a", basis="cc-pvtz-f12")

CH4_XYZ = "5\nch4 (converged opt)\nC 0 0 0\nH  0.629  0.629  0.629\nH -0.629 -0.629  0.629\nH -0.629  0.629 -0.629\nH  0.629 -0.629 -0.629"
OH_XYZ  = "2\noh (converged opt)\nO 0 0 0\nH 0 0 0.970"
CH3_XYZ = "4\nch3 (D3h)\nC 0 0 0\nH  1.078 0 0\nH -0.539  0.934 0\nH -0.539 -0.934 0"
H2O_XYZ = "3\nh2o (converged opt)\nO 0 0 0.117\nH 0 0.757 -0.469\nH 0 -0.757 -0.469"
TS_XYZ  = "7\nts (saddle, one imag mode)\nC 0 0 -1.0\nH 0.629 0.629 -0.371\nH -0.629 -0.629 -0.371\nH -0.629 0.629 -1.629\nH 0 0 0.300\nO 0 0 1.300\nH 0 0 2.270"

## 2 · Materialise tiny stand-in artifact files

Two artifact files — one TS-side, one CH3-side — so the example
exercises both buckets of `iter_calculation_entries(with_artifacts_only=True)`.


In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="tckdb-arc-style-nb-"))
(workdir / "ts_opt.log").write_text(
    "Entering Gaussian System, Link 0\n"
    "fake TS opt output log — converged saddle; one imag mode @ -1432 cm^-1\n"
)
(workdir / "ch3_sp.log").write_text(
    "Reading ORCA output stub\n"
    "fake CH3 high-LoT single-point\n"
)
artifact_paths = {
    "ts_opt": workdir / "ts_opt.log",
    "ch3_sp": workdir / "ch3_sp.log",
}
for k, p in artifact_paths.items():
    print(f"  {k:>8}: {p} ({p.stat().st_size} bytes)")

## 3 · Build the calculation trios

Each species (and the TS) gets a converged `opt` at the primary LoT,
a matching `freq` on the same geometry, and a high-LoT single-point.

The new `note=` kwarg on each factory carries workflow-side context
(`"lowest-energy converged structure; conformer search history
retained as artifacts."`). The note is preserved on the builder
(visible via `upload.iter_calculations()`) but is **not emitted on
the wire** — today's `CalculationInBundle` schema has no per-calc
note field.


In [ ]:
def calc_trio(label_prefix, xyz, final_e_h, sp_e_h, zpe_h,
              *, n_imag=0, imag_freq_cm1=None):
    geom = Geometry.from_xyz(xyz)
    opt = Calculation.opt(
        GAUSSIAN, LOT_OPT_FREQ, output_geometry=geom,
        final_energy_hartree=final_e_h, converged=True,
        label=f"{label_prefix} opt",
        note=f"{label_prefix}: lowest-energy converged structure; conformer search history retained as artifacts.",
    )
    fk = dict(n_imag=n_imag, zpe_hartree=zpe_h, depends_on=opt,
              label=f"{label_prefix} freq",
              note=f"{label_prefix}: harmonic on the converged opt geometry.")
    if imag_freq_cm1 is not None:
        fk["imag_freq_cm1"] = imag_freq_cm1
    freq = Calculation.freq(GAUSSIAN, LOT_OPT_FREQ, **fk)
    sp = Calculation.sp(
        ORCA, LOT_SP, electronic_energy_hartree=sp_e_h,
        depends_on=opt, label=f"{label_prefix} sp",
        note=f"{label_prefix}: high-LoT single point on the opt geometry.",
    )
    return opt, freq, sp

ch4_opt, ch4_freq, ch4_sp = calc_trio("ch4", CH4_XYZ, -40.518, -40.518, 0.0451)
oh_opt,  oh_freq,  oh_sp  = calc_trio("oh",  OH_XYZ,  -75.704, -75.711, 0.00853)
ch3_opt, ch3_freq, ch3_sp = calc_trio("ch3", CH3_XYZ, -39.838, -39.844, 0.0301)
h2o_opt, h2o_freq, h2o_sp = calc_trio("h2o", H2O_XYZ, -76.420, -76.428, 0.0214)
ts_opt,  ts_freq,  ts_sp  = calc_trio(
    "ts", TS_XYZ, final_e_h=-116.198, sp_e_h=-116.208, zpe_h=0.0521,
    n_imag=1, imag_freq_cm1=-1432.0,
)

ts_opt.add_artifact(artifact_paths["ts_opt"], kind="output_log")
ch3_sp.add_artifact(artifact_paths["ch3_sp"], kind="output_log")
print(f"opt.note example: {ch4_opt.note!r}")

## 4 · Source-calculation provenance with `SourceCalculations`

The kinetics record consumes a duplicate-role bag (two reactants → two
`reactant_energy` entries). CH3's thermo and statmech share one bag
and pick subsets via `.only(...)`.


In [ ]:
kin_sources = SourceCalculations(
    reactant_energy=[ch4_sp, oh_sp],
    product_energy=[ch3_sp, h2o_sp],
    ts_energy=ts_sp,
    freq=ts_freq,
)
kin = Kinetics.modified_arrhenius(
    A=1.93e13, A_units="cm3/mol/s",
    n=2.18, Ea=10.13, Ea_units="kJ/mol",
    Tmin=300, Tmax=2500,
    source_calculations=kin_sources.as_list(),
    label="CH4+OH H-abstraction, modified Arrhenius",
)
ch3_sources = SourceCalculations(opt=ch3_opt, freq=ch3_freq, sp=ch3_sp)
print(f"kinetics source calcs: {len(kin_sources)}")
print(f"ch3   source calcs:    {len(ch3_sources)}")

## 5 · Assemble the `ComputedReactionUpload`

Four species (CH4 / OH / CH3 / H2O), one TS with three calcs, and
per-species thermo + statmech on the CH3 product only.


In [ ]:
ch4 = Species(smiles="C",    charge=0, multiplicity=1, label="CH4")
oh  = Species(smiles="[OH]", charge=0, multiplicity=2, label="OH")
ch3 = Species(smiles="[CH3]",charge=0, multiplicity=2, label="CH3")
h2o = Species(smiles="O",    charge=0, multiplicity=1, label="H2O")

rxn = ChemReaction(
    reactants=[ch4, oh], products=[ch3, h2o],
    family="H_Abstraction",
    transition_state=TransitionState(
        charge=0, multiplicity=2,
        geometry=Geometry.from_xyz(TS_XYZ), label="ts",
    ),
    kinetics=[kin],
)

upload = ComputedReactionUpload(
    reaction=rxn,
    calculations=[ts_opt, ts_freq, ts_sp],
    species_calculations={
        ch4: [ch4_opt, ch4_freq, ch4_sp],
        oh:  [oh_opt,  oh_freq,  oh_sp],
        ch3: [ch3_opt, ch3_freq, ch3_sp],
        h2o: [h2o_opt, h2o_freq, h2o_sp],
    },
    species_thermo={
        ch3: Thermo.nasa(
            coeffs_low=[3.5] + [0.0] * 6, coeffs_high=[3.5] + [0.0] * 6,
            t_low=200, t_mid=1000, t_high=5000,
            h298_kj_mol=146.7, s298_j_mol_k=194.2,
            source_calculations=ch3_sources.only("opt", "freq", "sp"),
        ),
    },
    species_statmech={
        ch3: Statmech(
            external_symmetry=6, point_group="D3h",
            is_linear=False, rigid_rotor_kind="symmetric_top",
            statmech_treatment="rrho",
            source_calculations=ch3_sources.only("opt", "freq"),
        ),
    },
)
print(f"upload assembled — {len(list(upload.iter_calculations()))} total calcs")

## 6 · `upload.summary().to_text()` — preview surface

A small builder-side viewer. Section markers (Identity / Calculations
/ Scientific blocks / Artifacts / Diagnostics) are stable; exact
wording around them is not.


In [ ]:
print(upload.summary().to_text())

## 7 · Emission diagnostics

Every warning here flags data the builder accepted locally but
today's bundle schemas don't carry on the wire. The two
`artifact_upload_requires_second_phase` entries are the cue to run
the phase-2 plan below.


In [ ]:
diags = upload.emission_diagnostics()
if not diags:
    print("(none)")
for d in diags:
    print(f"[{d.level.upper():>7}] {d.code}")
    print(f"          path: {d.path}")
    msg = d.message.strip()
    for chunk in (msg[i:i+70] for i in range(0, len(msg), 70)):
        print(f"          {chunk}")

## 8 · Workflow → builder mapping

Walks `iter_calculation_entries()` to surface which bucket each calc
landed in, plus the source-calculation counts written by
`SourceCalculations`.


In [ ]:
by_bucket = {}
for entry in upload.iter_calculation_entries():
    by_bucket.setdefault(entry.bucket, []).append(entry.calculation)
for bucket, calcs in by_bucket.items():
    by_type = {}
    for c in calcs:
        by_type[c.type] = by_type.get(c.type, 0) + 1
    rendered = ", ".join(f"{k}={v}" for k, v in sorted(by_type.items()))
    print(f"  - [{bucket}]  {rendered}")

payload = upload.to_payload()
if payload.get("kinetics"):
    ksrc = payload["kinetics"][0].get("source_calculations", [])
    print(f"  - kinetics[0] source_calculations: {len(ksrc)} entries")
for sp in payload.get("species", []):
    tsrc = (sp.get("thermo")    or {}).get("source_calculations", [])
    smsrc = (sp.get("statmech") or {}).get("source_calculations", [])
    if tsrc or smsrc:
        print(
            f"  - species[{sp['key']}]: "
            f"thermo source_calculations={len(tsrc)}, "
            f"statmech source_calculations={len(smsrc)}"
        )

## 9 · Artifact summary

What's attached to which calculation, in which bucket. These files
don't move during the scientific upload — only metadata travels;
phase-2 ships the bytes once the server returns calculation IDs.


In [ ]:
entries = list(upload.iter_calculation_entries(with_artifacts_only=True))
if not entries:
    print("(no artifacts attached)")
else:
    total = sum(len(e.calculation.artifacts) for e in entries)
    print(f"{total} artifact(s) across {len(entries)} calculation(s):")
    for entry in entries:
        calc = entry.calculation
        for art in calc.artifacts:
            label = calc.label or calc.type
            print(f"  - [{entry.bucket:>4}] {label:<10} {art.kind:<14} {art.path}")

## 10 · Artifact plan preview (offline, mock IDs)

`upload.artifact_plan(result)` needs a real server response; the
public `artifact_plan_preview()` helper mints deterministic synthetic
IDs so producers can inspect the shape offline.


In [ ]:
plan_preview = upload.artifact_plan_preview()
for entry in plan_preview:
    print(
        f"  - calc_key={entry.calculation_key:<14} "
        f"calc_id={entry.calculation_id:<6} "
        f"kind={entry.kind:<12} path={entry.path}"
    )

## 11 · Truncated wire payload preview

What the bundle endpoint would receive. Note that **no artifact bytes
appear in the payload** — they ride on the second-phase POST.

Also note that the per-calc `note=` values supplied above do *not*
appear here — they're builder-local annotations (see the comment in
cell 06).


In [ ]:
rendered = json.dumps(payload, indent=2)
if len(rendered) > 1200:
    rendered = rendered[:1200] + "\n  …(truncated)"
print(rendered)

## 12 · Optional: live upload

The cell below performs the two-phase upload when both
`TCKDB_BASE_URL` and `TCKDB_API_KEY` are set. With either missing
the cell short-circuits and prints a friendly message — no network
call.


In [ ]:
base_url = os.environ.get("TCKDB_BASE_URL")
api_key  = os.environ.get("TCKDB_API_KEY")

if not base_url or not api_key:
    print("TCKDB_BASE_URL or TCKDB_API_KEY not set — skipping live upload.")
else:
    with TCKDBClient(base_url, api_key=api_key) as client:
        with warnings.catch_warnings():
            warnings.simplefilter("always")
            result = client.upload(upload, warn_on_dropped_fields=True)
        print("== Server response (truncated) ==")
        print(json.dumps(result, indent=2)[:1200])
        print()

        try:
            plan = upload.artifact_plan(result)
        except Exception as exc:
            print(f"artifact_plan failed: {exc}")
        else:
            idem_prefix = f"arc-style-nb:{int(time.time())}"
            print(f"== Artifact upload (phase 2, prefix={idem_prefix!r}) ==")
            if not plan:
                print("(no artifacts to upload)")
            else:
                results = client.upload_artifacts(plan, idempotency_key_prefix=idem_prefix)
                for entry, response in zip(plan, results):
                    n = len(response.get("artifacts", [])) if isinstance(response, dict) else 0
                    print(f"  - calc_id={entry.calculation_id:<6} "
                          f"kind={entry.kind:<12} -> {n} server row(s)")

---

**Where to go next**

- The matching script `builder_arc_style_dry_run.py` is the same
  flow without a Jupyter kernel; it's what
  `tests/test_builder_arc_style_dry_run.py` runs in CI.
- The full builder-layer spec lives at
  [`docs/builder_api_mvp.md`](../docs/builder_api_mvp.md).
- The conformer-boundary policy is at
  [`docs/conformer_semantic_boundary.md`](../docs/conformer_semantic_boundary.md).
